# Question & Answer with LangChain

- Some of the complex systems built are the ones that can answer questions on top or about a document. 
- LLM can answer questions using data from given texts.
- Combine language models with data that they were not trained on previously which makes it flexible and adaptible to your use case.
- Some key components of LangChain are embedding models and vector stores.

#### _Question Answering over Documents:_

Import these -
- RetrivalQA from chain which will do retrival over documents
- Language model
- CVS loader from document loaders
- Doc array in memory search from vector stores which is an in memory vector store and does not need to connect to external database - there are many types of vector stores
- display and markdown

    from langchain.chains import RetrievalQA
    from langchain.chat_models import ChatOpenAI
    from langchain.document_loaders import CSVLoader
    from langchain.vectorstores import DocArrayInMemorySearch
    from IPython.display import display, Markdown
    from langchain.llms import OpenAI

Load Example csv file:

    file = 'OutdoorClothingCatalog_1000.csv'
    loader = CSVLoader(file_path=file)

Import vector store index creator from indexes to create a vector store easily:

    from langchain.indexes import VectorstoreIndexCreator

Specify vector store class as docs array in memory search and take in list of documents to be loaded :

    index = VectorstoreIndexCreator(
        vectorstore_cls=DocArrayInMemorySearch
    ).from_loaders([loader])

Ask Queries about Document:

    query ="Please list all your shirts with sun protection \
    in a table in markdown and summarize each one."

    llm_replacement_model = OpenAI(temperature=0, 
                                model='gpt-3.5-turbo-instruct')

    response = index.query(query, 
                        llm = llm_replacement_model)

Display reponse:

    display(Markdown(response))

_Output evidence:_

    print(response)




| Name | Description | Sun Protection Rating |
| --- | --- | --- |
| Men's Tropical Plaid Short-Sleeve Shirt | Made of 100% polyester, UPF 50+ rating, wrinkle-resistant, front and back cape venting, two front bellows pockets | SPF 50+, blocks 98% of harmful UV rays |
| Men's Plaid Tropic Shirt, Short-Sleeve | Made of 52% polyester and 48% nylon, UPF 50+ rating, SunSmart technology, wrinkle-free, front and back cape venting, two front bellows pockets | SPF 50+, blocks 98% of harmful UV rays |
| Men's TropicVibe Shirt, Short-Sleeve | Made of 71% nylon and 29% polyester, UPF 50+ rating, front and back cape venting, two front bellows pockets | SPF 50+, blocks 98% of harmful UV rays |
| Sun Shield Shirt | Made of 78% nylon and 22% Lycra Xtra Life fiber, UPF 50+ rating, moisture-wicking, abrasion-resistant, fits over swimsuit | SPF 50+, blocks 98% of harmful UV rays |

#### _LLMs on Documents:_

Since LLM can only inspect around 1000 words at a time, embedding and vector stores help to find answers from huge documents.

Embedding:
- It creates numerical representation over texts
- captures semantic meaning of the piece of text 
- pieces of text with similar content will have similar vectors
- Helps find pieces of text that are like each other
- useful to know what pieces of text to pass to model when answering questions


Vector database:
- To store vector representations created
- Add chunks of texts into a vector database from documents
- Break document to pieces of text that are smaller than actual document with relevant text
- Embed each chuck of texts and store them in a vector database by creating index
- Can easily find pieces of text during runtime that are relevant to a query
- The query is first embeded and then compared to all the vectors in the database and pick the ones most similar

Load Example csv file:

    from langchain.document_loaders import CSVLoader
    loader = CSVLoader(file_path=file)

Load documents:

    docs = loader.load()

Create embeddings with small chunks of text using open ai embeddings:

    from langchain.embeddings import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings()

Create embeddings for all documents and store in the doc array in memory search vector store:

    db = DocArrayInMemorySearch.from_documents(
        docs, 
        embeddings
    )

Use vector store to find pieces of text similar to a query:

    query = "Please suggest a shirt with sunblocking"

Use similarity search method on the vector database with the query:

    docs = db.similarity_search(query)

=> <b>len(docs) = 4 which means 4 pages of the document are similar to the query</b>

Retriver:
- It is an interface that takes in a query and returns documents
- Many methods of retrival - simple or complex methods
- Do question answering over documents


Retrival QA Chain:
- This chain will retrive the documents and answers questions based on the query
- Pass language model to get text generation of the answer
- Pass chain type as stuffs which is the simplest method that stuffs all the documents into contect of the model in one call. There are other methods.
- Pass a retriever to fetch documents

    retriever = db.as_retriever()

    llm = ChatOpenAI(temperature = 0.0, model=llm_model)

    qdocs = "".join([docs[i].page_content for i in range(len(docs))])

    response = llm.call_as_llm(f"{qdocs} Question: Please list all your \
    shirts with sun protection in a table in markdown and summarize each one.") 

    qa_stuff = RetrievalQA.from_chain_type(
        llm=llm, 
        chain_type="stuff", 
        retriever=retriever, 
        verbose=True
    )

    query =  "Please list all your shirts with sun protection in a table \
    in markdown and summarize each one."

Run Retrival QA Chain with query:

    response = qa_stuff.run(query)

#### _Q&A over Lots of Chunks of Data:_

- Stuffs method will not be possible as LLMs have a limited context length
- Methods that can be used are:
    1. Map-reduce - passes all the chucks with the question to LLMs and gets back a response. Then it uses another LLM to summarise all the responses from different LLMs to get final response.
    2. Refine - Also takes chunks with the question to LLM iteratively. Builds upon the answer from previous LLM response. Combines prev LLM response and new chunk of data in iteration
    3. Map rerank - Call LLM for each chunk and ask to return a score and return highest score if its relevant to the document. Still expensive as it makes many LLM calls
- Most common methods are stuffs and map reduce methods
- These methods can also be used for summarisation